In [ ]:
import pandas as pd

file_path = "/Users/mali/Desktop/UW/Climate Risk/WA OR CA Utility.xlsx"
xls = pd.ExcelFile(file_path)

df = pd.read_excel(xls, sheet_name='Summary', header=3)

df.dropna(how='all', inplace=True)

if df.iloc[0].str.contains("Issuer Name").any():
    df.columns = df.iloc[0]
    df = df.drop(index=0)

df_util = df[df['Issuer Name'].astype(str).str.contains("Pub Util Dist", na=False)]

print(df_util.head())


In [ ]:
import json
import pandas as pd

json_path = "/Users/mali/Desktop/UW/Climate Risk/county_to_service_area.json"
with open(json_path, 'r') as file:
    county_to_service_area = json.load(file)

df_service_area = pd.DataFrame(list(county_to_service_area.items()), columns=["Issuer Name", "Service Area"])

print(df_service_area)


In [ ]:
import json

with open("/Users/mali/Desktop/UW/Climate Risk/county_to_service_area.json", "r") as f:
    county_to_service_area = json.load(f)

df_util['Service Area'] = df_util['Issuer Name'].map(county_to_service_area)

print(df_util[['Issuer Name', 'Service Area']].head())


In [ ]:
import pandas as pd

file_path = "/Users/mali/Desktop/UW/Climate Risk/msrb_transaction.csv.gz"
df_msrb = pd.read_csv(file_path)

print(df_msrb.head(10))

print(df_msrb.columns)


In [ ]:
print(df_util.columns.tolist())

In [ ]:
df_merged = df_msrb.merge(
    df_util[['9-digit CUSIP', 'Amout Outstanding', 'Call Flag', 'Insurance Flag', 'Service Area']],
    left_on='cusip',
    right_on='9-digit CUSIP',
    how='inner'
)

print(df_merged.head())

In [ ]:
df_merged['maturity_date'] = pd.to_datetime(df_merged['maturity_date'])
df_merged['trade_date'] = pd.to_datetime(df_merged['trade_date'])

df_merged['years_to_maturity'] = (df_merged['maturity_date'] - df_merged['trade_date']).dt.days / 365.25
df_merged['years_to_maturity'] = df_merged['years_to_maturity'].round()

print(df_merged.head())

In [ ]:
df_benchmark = pd.read_csv("/Users/mali/Desktop/UW/Climate Risk/Tradeweb_MUNI_data.csv")
df_benchmark.rename(columns={df_benchmark.columns[0]: "trade_date"}, inplace=True)
df_benchmark['trade_date'] = pd.to_datetime(df_benchmark['trade_date'])

df_benchmark_long = df_benchmark.melt(
    id_vars="trade_date",
    var_name="years_to_maturity",
    value_name="benchmark_yield"
)

df_benchmark_long['years_to_maturity'] = df_benchmark_long['years_to_maturity'].str.extract(r'(\d+)').astype(int)

print(df_benchmark_long)

In [ ]:
df_final = df_merged.merge(
    df_benchmark_long,
    on=['trade_date', 'years_to_maturity'],
    how='left'
)

df_final['spread'] = df_final['yield'] - df_final['benchmark_yield']

print(df_final)
print(df_final.columns.tolist())

In [ ]:
# Filter:
df_filtered = df_final[
    (df_final['yield'] >= 0) &                              # Exclude negative yields
    (df_final['dollar_price'] <= 150) &                     # Exclude prices above $150
    (df_final['dollar_price'] >= 50) &                      # Exclude prices below $50
    (df_final['coupon'] <= 20) &                            # Exclude coupon rates > 20%
    (df_final['years_to_maturity'] <= 30)                   # Exclude time-to-maturity > 30 years
]

# Filter to only include bonds traded at least 10 times
trade_counts = df_filtered['cusip'].value_counts()
valid_cusips = trade_counts[trade_counts >= 10].index
df_filtered = df_filtered[df_filtered['cusip'].isin(valid_cusips)]

df_filtered = df_filtered.reset_index(drop=True)

print(df_filtered)


In [ ]:
import pandas as pd

file_path = "/Users/mali/Desktop/UW/Climate Risk/utility_service_area_fire_metrics.csv.gz"
df_fwi = pd.read_csv(file_path)

print(df_fwi.head())

print(df_fwi.columns)

In [ ]:
import pandas as pd

file_path = "/Users/mali/Desktop/UW/Climate Risk/utility_service_area_average_adjusted_gross_income.csv"
df_income = pd.read_csv(file_path)

print(df_income.head())

In [ ]:
df_filtered['Service Area'] = df_filtered['Service Area'].str.strip().str.upper()
df_fwi['NAME'] = df_fwi['NAME'].str.strip().str.upper()

df_merge = df_filtered.merge(
    df_fwi,
    left_on='Service Area',
    right_on='NAME',
    how='left'
)

df_merge = df_merge[(df_merge['CUSTOMERS'] > 0)]
df_merge['trade_date'] = pd.to_datetime(df_merge['trade_date'], errors='coerce')
df_merge['Ig_Date'] = pd.to_datetime(df_merge['Ig_Date'], errors='coerce')

df_merge['days_since_ignition'] = (df_merge['trade_date'] - df_merge['Ig_Date']).dt.days
df_merge.loc[df_merge['days_since_ignition'] < 0, 'days_since_ignition'] = pd.NA

df_merge['NAME'] = df_merge['NAME'].str.strip().str.upper()
df_income['NAME'] = df_income['NAME'].str.strip().str.upper()

df_merge['trade_year'] = df_merge['trade_date'].dt.year

df_merge = df_merge.merge(
    df_income,
    left_on=['NAME', 'trade_year'],
    right_on=['NAME', 'year'],
    how='left'
)

income_2014 = df_income[df_income['year'] == 2014].set_index('NAME')['average_adjusted_gross_income']
income_2022 = df_income[df_income['year'] == 2022].set_index('NAME')['average_adjusted_gross_income']

mask_after_2022 = (df_merge['trade_year'] > 2022) & (df_merge['average_adjusted_gross_income'].isna())
df_merge.loc[mask_after_2022, 'average_adjusted_gross_income'] = df_merge.loc[mask_after_2022, 'NAME'].map(income_2022)

mask_before_2014 = (df_merge['trade_year'] < 2014) & (df_merge['average_adjusted_gross_income'].isna())
df_merge.loc[mask_before_2014, 'average_adjusted_gross_income'] = df_merge.loc[mask_before_2014, 'NAME'].map(income_2014)

print(df_merge)

In [ ]:
df_merge['trade_date'] = pd.to_datetime(df_merge['trade_date'], errors='coerce')
df_merge['Ig_Date'] = pd.to_datetime(df_merge['Ig_Date'], errors='coerce')

df_merge['days_since_ignition'] = (df_merge['trade_date'] - df_merge['Ig_Date']).dt.days

trades_within_30 = df_merge[
    (df_merge['days_since_ignition'] >= 0) &
    (df_merge['days_since_ignition'] <= 30)
]

df_merge = trades_within_30.copy()

print(f"Number of trades within 30 days after ignition: {len(trades_within_30)}")
print(df_merge)


In [ ]:
df_merge['trade_date'] = pd.to_datetime(df_merge['trade_date'], errors='coerce')
df_merge['Ig_Date'] = pd.to_datetime(df_merge['Ig_Date'], errors='coerce')

df_merge['days_since_ignition'] = (df_merge['trade_date'] - df_merge['Ig_Date']).dt.days

valid_trades = df_merge[df_merge['days_since_ignition'] >= 0]

first_trades = valid_trades.sort_values(['Service Area', 'Ig_Date', 'days_since_ignition'])
first_trades = first_trades.groupby(['Service Area', 'Ig_Date']).first().reset_index()

print(f"Number of first trades captured: {len(first_trades)}")

In [ ]:
columns_to_keep = [
    "NAME",
    "9-digit CUSIP",
    "trade_date",
    "spread",
    "par_traded",
    "Call Flag",
    "Insurance Flag",
    "CUSTOMERS",
    "average_adjusted_gross_income",
    "fire_severity_mean",
    "fire_severity_max",
    "BurnBndAc",
    "fwi_mean",
    "fwi_max",
    "days_since_ignition"
]

df_final_analysis = df_merge[columns_to_keep].copy()

df_final_analysis.to_csv("df_final_analysis.csv.gz", index=False, compression='gzip')

print("Final dataset prepared with shape:", df_final_analysis.shape)
print(df_final_analysis)


In [ ]:
na_counts = df_final_analysis.isna().sum()

print("Missing values per column:")
print(na_counts[na_counts > 0])

In [ ]:
df_final_analysis_clean = df_final_analysis.dropna().copy()

df_final_analysis_clean.to_csv("df_final_analysis_clean.csv.gz", index=False, compression='gzip')
print(f"Rows before dropping NA: {len(df_final_analysis)}")
print(f"Rows after dropping NA: {len(df_final_analysis_clean)}")

In [ ]:
pip install statsmodels

In [4]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("/Users/ericcollins/Downloads/df_final_analysis_clean.csv")

df = df.rename(columns={
    "Call Flag": "call_flag",
    "Insurance Flag": "insurance_flag",
    "9-digit CUSIP": "cusip9",
    "average_adjusted_gross_income": "income",
    "fire_severity_mean": "fire_sev_mean",
    "fire_severity_max": "fire_sev_max",
    "fwi_mean": "fwi_mean",
    "fwi_max": "fwi_max",
    "BurnBndAc": "burned_acres",
    "par_traded": "par_traded",
    "CUSTOMERS": "customers",
    "days_since_ignition": "days_since_ignition",
    "spread": "spread"
})

df['call_flag'] = df['call_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)
df['insurance_flag'] = df['insurance_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)

df['utility_id'] = df['NAME'].astype('category').cat.codes
df['time_id'] = pd.to_datetime(df['trade_date']).dt.to_period('M').astype(str)

dependent = 'spread'
base_controls = [
    "par_traded", "call_flag", "insurance_flag", "customers",
    "income", "burned_acres", "days_since_ignition"
]

fire_combinations = [
    ["fire_sev_mean", "fwi_mean"],
    ["fire_sev_mean", "fwi_max"],
    ["fire_sev_max", "fwi_mean"],
    ["fire_sev_max", "fwi_max"]
]

results = []
for severity, fwi in fire_combinations:
    formula = (
        f"{dependent} ~ {' + '.join(base_controls + [severity, fwi])} "
        "+ C(utility_id) + C(time_id)"
    )
    model = smf.ols(formula=formula, data=df).fit()
    results.append({
        "Model": f"{severity} + {fwi}",
        "R-squared": model.rsquared,
        "Adj. R-squared": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic
    })

summary_df = pd.DataFrame(results)
print(summary_df.head(5))


                      Model  R-squared  Adj. R-squared          AIC  \
0  fire_sev_mean + fwi_mean   0.364238        0.352318  2412.102520   
1   fire_sev_mean + fwi_max   0.364990        0.353084  2410.172433   
2   fire_sev_max + fwi_mean   0.362580        0.350628  2416.351492   
3    fire_sev_max + fwi_max   0.363743        0.351814  2413.371645   

           BIC  
0  2579.407927  
1  2577.477839  
2  2583.656898  
3  2580.677051  


In [5]:
import pandas as pd
import statsmodels.formula.api as smf

df['call_flag'] = df['call_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)
df['insurance_flag'] = df['insurance_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)

df['utility_id'] = df['NAME'].astype('category').cat.codes
df['time_id'] = pd.to_datetime(df['trade_date']).dt.to_period('M').astype(str)
df['time_id'] = df['time_id'].str.replace('-', '')  # "2023-09" -> "202309"

base_controls = [
    "par_traded", "call_flag", "insurance_flag", "customers",
    "income", "burned_acres", "days_since_ignition"
]

fire_combinations = [
    ["fire_sev_mean", "fwi_mean"],
    ["fire_sev_mean", "fwi_max"],
    ["fire_sev_max", "fwi_mean"],
    ["fire_sev_max", "fwi_max"]
]

def run_backward_elimination(df, formula, dep_var="spread", threshold=0.05):
    model = smf.ols(formula=formula, data=df).fit()
    terms = model.model.exog_names
    terms = [t for t in terms if t not in ['Intercept'] and not t.startswith('C(')]

    while True:
        model = smf.ols(formula='{} ~ {}'.format(dep_var, ' + '.join(terms) + ' + C(utility_id) + C(time_id)'), data=df).fit()
        pvals = model.pvalues.drop(labels=['Intercept'], errors='ignore')

        # Filter only original terms (exclude fixed effects)
        worst = pvals[[term for term in pvals.index if term in terms and not term.startswith('C(')]]
        worst = worst[worst > threshold]

        if worst.empty:
            break

        worst_var = worst.idxmax()
        print(f"Removing: {worst_var} (p = {worst[worst_var]:.4f})")
        terms.remove(worst_var)

    final_formula = '{} ~ {}'.format(dep_var, ' + '.join(terms) + ' + C(utility_id) + C(time_id)')
    return smf.ols(formula=final_formula, data=df).fit()

summary_rows = []

for severity, fwi in fire_combinations:
    print(f"\n Running model for: {severity} + {fwi}")
    
    all_vars = base_controls + [severity, fwi]
    full_formula = f"spread ~ {' + '.join(all_vars)} + C(utility_id) + C(time_id)"
    
    full_model = smf.ols(formula=full_formula, data=df).fit()
    
    refined_model = run_backward_elimination(df, full_formula)
    
    summary_rows.append({
        "Model": f"{severity} + {fwi}",
        "Base R²": round(full_model.rsquared, 4),
        "Base AIC": round(full_model.aic, 2),
        "Base BIC": round(full_model.bic, 2),
        "Refined R²": round(refined_model.rsquared, 4),
        "Refined AIC": round(refined_model.aic, 2),
        "Refined BIC": round(refined_model.bic, 2)
    })

summary_df = pd.DataFrame(summary_rows)
print("\n Final Model Comparison Summary:\n")
print(summary_df.to_string(index=False))




 Running model for: fire_sev_mean + fwi_mean
Removing: insurance_flag (p = 0.9038)
Removing: income (p = 0.6184)
Removing: call_flag (p = 0.7373)
Removing: fwi_mean (p = 0.6103)
Removing: burned_acres (p = 0.1461)

 Running model for: fire_sev_mean + fwi_max
Removing: burned_acres (p = 0.9094)
Removing: income (p = 0.4225)
Removing: insurance_flag (p = 0.6128)
Removing: fire_sev_mean (p = 0.0651)

 Running model for: fire_sev_max + fwi_mean
Removing: burned_acres (p = 0.9603)
Removing: income (p = 0.7436)
Removing: fire_sev_max (p = 0.5353)
Removing: fwi_mean (p = 0.0737)

 Running model for: fire_sev_max + fwi_max
Removing: fire_sev_max (p = 0.8426)
Removing: call_flag (p = 0.6907)
Removing: income (p = 0.4743)
Removing: burned_acres (p = 0.5456)

 Final Model Comparison Summary:

                   Model  Base R²  Base AIC  Base BIC  Refined R²  Refined AIC  Refined BIC
fire_sev_mean + fwi_mean   0.3642   2412.10   2579.41      0.3632      2408.77      2559.89
 fire_sev_mean + fwi_m

In [ ]:
pip install linearmodels

In [1]:
%reset -f

import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

In [ ]:
df = pd.read_csv("/Users/ericcollins/Downloads/df_final_analysis_clean.csv")

df = df.rename(columns={
    "Call Flag": "call_flag",
    "Insurance Flag": "insurance_flag",
    "9-digit CUSIP": "cusip9",
    "average_adjusted_gross_income": "income",
    "fire_severity_mean": "fire_sev_mean",
    "fire_severity_max": "fire_sev_max",
    "fwi_mean": "fwi_mean",
    "fwi_max": "fwi_max",
    "BurnBndAc": "burned_acres",
    "par_traded": "par_traded",
    "CUSTOMERS": "customers",
    "days_since_ignition": "days_since_ignition",
    "spread": "spread"
})

df['call_flag'] = df['call_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)
df['insurance_flag'] = df['insurance_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)

df['utility_id'] = df['NAME'].astype('category').cat.codes
df['time_id'] = pd.to_datetime(df['trade_date']).dt.to_period('M').dt.to_timestamp()
df = df.set_index(['utility_id', 'time_id'])

controls = ["par_traded", "call_flag", "insurance_flag", "customers", "income", "burned_acres", "days_since_ignition"]

fire_combinations = [
    ("fire_sev_mean", "fwi_mean"),
    ("fire_sev_mean", "fwi_max"),
    ("fire_sev_max", "fwi_mean"),
    ("fire_sev_max", "fwi_max")
]

results = []

for fire, fwi in fire_combinations:
    df['fire'] = df[fire]
    df['fwi'] = df[fwi]
    df['fire_fwi_interaction'] = df['fire'] * df['fwi']

    exog_vars = controls + ['fire', 'fwi', 'fire_fwi_interaction']
    exog = df[exog_vars].copy()

    exog = exog.loc[:, exog.apply(pd.Series.nunique) > 1]

    if 'const' not in exog.columns:
        exog.insert(0, 'const', 1.0)

    model = PanelOLS(
        df['spread'], exog,
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True,
        check_rank=False
    )

    fit_result = model.fit(cov_type='robust')

    # Extract robust SEs and p-values
    coefs = fit_result.params
    se = fit_result.std_errors
    pvals = fit_result.pvalues

# Format them into readable dicts (or leave as full Series)
    coef_summary = {
        var: {
            "coef": round(coefs[var], 4),
            "std_err": round(se[var], 4),
            "p_value": round(pvals[var], 4)
        } for var in coefs.index
    }

    results.append({
        "Model": f"{fire} + {fwi}",
        "R-squared (within)": round(fit_result.rsquared, 4),
        "Coefficients": coef_summary
    })


summary_df = pd.DataFrame(results)
print("\n Panel Regression Model Summary:\n")
print(summary_df.to_string(index=False))


 Panel Regression Model Summary:

                   Model  R-squared (within)                                                                                                                                                                                                                                                                                                                                                                                                   Coefficients
fire_sev_mean + fwi_mean              0.0378  {'const': {'coef': -0.4633, 'std_err': 2.5169, 'p_value': 0.854}, 'par_traded': {'coef': -0.0, 'std_err': 0.0, 'p_value': 0.0001}, 'income': {'coef': 0.0, 'std_err': 0.0, 'p_value': 0.8354}, 'burned_acres': {'coef': -0.0, 'std_err': 0.0, 'p_value': 0.1731}, 'days_since_ignition': {'coef': 0.0137, 'std_err': 0.0021, 'p_value': 0.0}, 'fire': {'coef': 0.1762, 'std_err': 0.0661, 'p_value': 0.0078}}
 fire_sev_mean + fwi_max              0.0378  {'const': {'coef': -0.4633, 'st

/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/1284981304.py:58: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  fit_result = model.fit(cov_type='robust')
/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/1284981304.py:58: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  fit_result = model.fit(cov_type='robust')
/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/1284981304.py:58: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  fit_result = model.fit(cov_type='robust')
/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/1284981304.py:58: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  fit_result = model.fit(cov_type='robust')


In [3]:
import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

df = pd.read_csv("/Users/ericcollins/Downloads/df_final_analysis_clean.csv")

df = df.rename(columns={
    "Call Flag": "call_flag",
    "Insurance Flag": "insurance_flag",
    "9-digit CUSIP": "cusip9",
    "average_adjusted_gross_income": "income",
    "fire_severity_mean": "fire_sev_mean",
    "fire_severity_max": "fire_sev_max",
    "fwi_mean": "fwi_mean",
    "fwi_max": "fwi_max",
    "BurnBndAc": "burned_acres",
    "par_traded": "par_traded",
    "CUSTOMERS": "customers",
    "days_since_ignition": "days_since_ignition",
    "spread": "spread"
})

df['call_flag'] = df['call_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)
df['insurance_flag'] = df['insurance_flag'].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)

df['utility_id'] = df['NAME'].astype('category').cat.codes
df['time_id'] = pd.to_datetime(df['trade_date']).dt.to_period('M').dt.to_timestamp()
df = df.set_index(['utility_id', 'time_id'])

controls = ["par_traded", "call_flag", "insurance_flag", "customers", "income", "burned_acres", "days_since_ignition"]
fire_combinations = [
    ("fire_sev_mean", "fwi_mean"),
    ("fire_sev_mean", "fwi_max"),
    ("fire_sev_max", "fwi_mean"),
    ("fire_sev_max", "fwi_max")
]

results = []

for fire, fwi in fire_combinations:
    df['fire'] = df[fire]
    df['fwi'] = df[fwi]
    df['fire_fwi_interaction'] = df['fire'] * df['fwi']

    exog_vars = controls + ['fire', 'fwi', 'fire_fwi_interaction']
    exog = df[exog_vars].copy()
    exog = exog.loc[:, exog.apply(pd.Series.nunique) > 1]
    if 'const' not in exog.columns:
        exog.insert(0, 'const', 1.0)

    full_model = PanelOLS(df['spread'], exog,
                          entity_effects=True,
                          time_effects=True,
                          drop_absorbed=True,
                          check_rank=False)
    full_result = full_model.fit(cov_type='robust')

    pvals = full_result.pvalues
    sig_vars = [var for var in pvals.index if pvals[var] <= 0.05]

    if len(sig_vars) < 2:
        print(f"⚠️ Model {fire} + {fwi} skipped (not enough significant variables).")
        continue

    exog_reduced = exog[sig_vars]
    if 'const' not in exog_reduced.columns:
        exog_reduced.insert(0, 'const', 1.0)

    reduced_model = PanelOLS(df['spread'], exog_reduced,
                             entity_effects=True,
                             time_effects=True,
                             drop_absorbed=True,
                             check_rank=False)
    reduced_result = reduced_model.fit(cov_type='robust')

    results.append({
        "Model": f"{fire} + {fwi}",
        "Included Vars": sig_vars,
        "R-squared": round(reduced_result.rsquared, 4)
    })

# Show results
summary_df = pd.DataFrame(results)
print("\n Refitted Models After p-value Filtering:\n")
print(summary_df.to_string(index=False))



 Refitted Models After p-value Filtering:

                   Model                                                      Included Vars  R-squared
fire_sev_mean + fwi_mean [par_traded, days_since_ignition, fire, fwi, fire_fwi_interaction]     0.0409
 fire_sev_mean + fwi_max [par_traded, days_since_ignition, fire, fwi, fire_fwi_interaction]     0.0429
 fire_sev_max + fwi_mean                                  [par_traded, days_since_ignition]     0.0334
  fire_sev_max + fwi_max                                  [par_traded, days_since_ignition]     0.0334


/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/2619847902.py:56: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  full_result = full_model.fit(cov_type='robust')
/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/2619847902.py:56: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  full_result = full_model.fit(cov_type='robust')
/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/2619847902.py:56: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  full_result = full_model.fit(cov_type='robust')
/var/folders/23/bwtd0x3j2qjf60tdmj7g56400000gn/T/ipykernel_96158/2619847902.py:56: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

customers

  full_result = full_model.fit(cov_type='robust')


In [7]:
df.head(50)

NAME     cusip9  trade_date  \
utility_id time_id                                                        
1          2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-06-22   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-06-22   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-06-22   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-06-22   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-06-22   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-06-22   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-09-14   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-09-22   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-09-22   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-09-26   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869LG1  2023-09-26   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2022-07-29   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2022-07-29   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2023-06-26   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2023-06-26   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2023-06-26   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2023-06-26   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2023-06-27   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869LH9  2023-06-27   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869MR6  2023-09-22   
           2023-10-01  PUD NO 1 OF BENTON COUNTY  082869MR6  2023-10-02   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2022-10-19   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2022-10-19   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2022-10-24   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2022-10-24   
           2023-07-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2023-07-11   
           2023-07-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2023-07-11   
           2023-07-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2023-07-12   
           2023-07-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2023-07-12   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2023-09-22   
           2023-09-01  PUD NO 1 OF BENTON COUNTY  082869MT2  2023-09-22   
           2023-07-01  PUD NO 1 OF BENTON COUNTY  082869MV7  2023-07-12   
           2023-07-01  PUD NO 1 OF BENTON COUNTY  082869MV7  2023-07-12   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869MW5  2022-07-11   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869MW5  2022-07-11   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869MW5  2022-07-11   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-07-05   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-07-05   
           2022-07-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-07-26   
           2022-08-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-08-03   
           2022-08-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-08-03   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-10-07   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-10-07   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-10-13   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-10-13   
           2022-10-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2022-10-13   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2023-06-20   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2023-06-20   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2023-06-20   
           2023-06-01  PUD NO 1 OF BENTON COUNTY  082869NE4  2023-06-20   

                       spread  par_traded  call_flag  insurance_flag  \
utility_id time_id                                                     